# LangChain Agent 스켈레톤 구현(OpenAI)

In [1]:
!pip show langchain langchain-core | grep -E "Name: |Version:"
## 2511
# Name: langchain
# Version: 0.3.27
# Name: langchain-core
# Version: 0.3.79
## 2601
# Name: langchain
# Version: 1.2.4
# Name: langchain-core
# Version: 1.2.7

Name: langchain
Version: 1.2.4
Name: langchain-core
Version: 1.2.7


In [2]:
#!pip install langchain==0.3.27 langchain-core==0.3.79  -q

# Colab 요구사항에 맞게 requests 강제 고정(의존성 무시)
!pip install -q --no-deps "requests==2.32.4" -q
!pip install --prefer-binary langchain-community==0.3.30 langchain-openai==0.3.33 -q
# google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.9/458.9 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
langgraph-prebuilt 1.0.6 requires langchain-core>=1.0.0, but you have langchain-core 0.3.83 which is incompatible.


In [3]:
import os
from google.colab import userdata

# Colab Secrets에서 토큰 읽어오기
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

# OpenAI API 키
os.environ["OPENAI_API_KEY"] = "sk-proj-YOUR_KEY_HERE"
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
!hf auth whoami

user:  gshong
orgs:  LLM2506,aicmap


Langchain OpenAI API  : https://python.langchain.com/api_reference/openai/index.html

In [4]:
import torch
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.agents import initialize_agent, Tool, AgentType
from langchain_core.messages import SystemMessage

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

## 매뉴얼 파일 생성


In [5]:
# 매뉴얼 파일 생성
manual_content = """
= DUV (Deep Ultraviolet) 공정 매뉴얼 =

문서 버전: 1.2
작성일: 2025-06-12

== 1. 개요 ==
DUV 리소그래피는 248nm 또는 193nm 파장의 빛을 사용하여 웨이퍼에 미세 회로 패턴을 형성하는 핵심 공정입니다.

== 2. 주요 단계 ==
2.1 웨이퍼 준비 (Wafer Preparation)
- 웨이퍼 세척 및 표면 처리

2.2 감광액 도포 (PR Coating)
- 스핀 코팅 방식을 사용하여 균일한 두께의 감광액(PR) 막을 형성합니다.

2.3 노광 (Exposure)
- 마스크에 설계된 패턴을 빛을 이용해 웨이퍼의 감광액 위로 전사시킵니다.
- **핵심 파라미터:** 노광 에너지(Dose), 초점(Focus)이 패턴의 정밀도를 결정합니다.
- 노광 장비의 정렬(Align) 정확도가 수율에 큰 영향을 미칩니다.

2.4 현상 (Development)
- 노광된 영역 또는 노광되지 않은 영역의 감광액을 선택적으로 제거하여 패턴을 완성합니다.
"""

with open("DUV_manual.txt", "w") as f:
    f.write(manual_content)

print("'DUV_manual.txt' 매뉴얼 파일 생성.")

def load_manual(path: str) -> str:
    """지정된 경로(path)의 텍스트 파일을 읽어 그 내용을 문자열로 반환합니다."""
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        return f"오류: '{path}' 파일을 찾을 수 없습니다."

'DUV_manual.txt' 매뉴얼 파일 생성.


## 반도체 데이터베이스 정의


In [6]:
# Tool 구현에 필요한 모든 데이터와 함수 정의

# 반도체 데이터베이스 정의
semiconductor_processes = {
    "포토리소그래피": {
        "주요파라미터": "노광량(Exposure Dose), 초점(Focus), 감광액(PR) 두께",
        "설명": "반도체 웨이퍼 위에 회로 패턴을 형성하는 핵심 공정",
        "단위": "노광량(mJ/cm²), 초점(μm), PR두께(nm)",
        "중요도": "패턴 해상도와 정밀도를 결정하는 가장 중요한 공정"
    },
    "식각": {
        "주요파라미터": "가스 유량, 압력(Pressure), RF 파워, 식각 시간",
        "설명": "불필요한 물질을 제거하여 원하는 패턴을 만드는 공정",
        "단위": "가스유량(sccm), 압력(mTorr), RF파워(W), 시간(min)",
        "중요도": "선폭 제어와 프로파일 형성에 핵심적"
    },
    "증착": {
        "주요파라미터": "온도, 압력, 전구체(Precursor) 유량",
        "설명": "웨이퍼 위에 얇은 막을 형성하는 공정",
        "단위": "온도(°C), 압력(Torr), 유량(sccm)",
        "중요도": "막질과 두께 균일성을 결정"
    }
}

# Tool로 사용할 함수들 정의
def process_lookup(query: str) -> str:
    """
    사용자 질문(query)에서 공정 이름을 찾아, 해당 공정의 주요 파라미터, 설명, 단위, 중요도 등 상세 정보를 반환합니다.
    '포토리소그래피', '식각', '증착' 공정에 대한 정보를 조회할 수 있습니다.
    """
    print(f"🔍 [고급 검색] '{query}' 분석 중...")
    for process_name, info in semiconductor_processes.items():
        if process_name in query:
            detailed_info = f'''
====== {process_name} 공정 상세 정보 ======
- 주요 파라미터: {info['주요파라미터']}
- 측정 단위: {info['단위']}
- 공정 설명: {info['설명']}
- 중요도: {info['중요도']}
==========================================
'''
            print(f"[검색 성공] {process_name} 공정 정보 발견")
            return detailed_info
    available = ", ".join(semiconductor_processes.keys())
    result = f"해당 공정을 찾을 수 없습니다. 사용 가능한 공정: {available}"
    print(f"[검색 실패] {result}")
    return result

## Tool 리스트 정의


In [7]:
# 2개의 Tool로 구성된 리스트 정의
tools = [
    Tool(
        name="Semiconductor Process DB Lookup",
        func=process_lookup,
        description="반도체 공정(포토리소그래피, 식각, 증착)에 대한 상세 정보(주요 파라미터, 설명, 단위, 중요도)를 찾을 때 사용합니다. 사용자 질문에 공정 이름이 포함되어 있어야 합니다."
    ),
    Tool(
        name="Manual Loader",
        func=load_manual,
        description="로컬 텍스트 파일 형식의 매뉴얼을 읽을 때 사용합니다. 'DUV_manual.txt'와 같이 정확한 파일 경로를 입력해야 합니다."
    )
]


## LLM 래퍼 설정  
> Wrapper는 LLM의 종류와 특성을 LangChain에 맞도록 숨겨주는 역할을 함, Interface 통일, LLM 교체를 쉽게 할 수 있음     
> OpenAI의 GPT-4 모델을 사용합니다. temperature=0으로 설정하여 일관성 있는 답변을 유도합니다.  

>model_name: https://platform.openai.com/docs/models?utm_source=chatgpt.com

In [8]:
from langchain_openai import ChatOpenAI
# ChatOpenAI(): OpenAI API를 langchain에 맞춰놓은 Wrapping API임
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 시스템 프롬프트 내용 정의 : 역할과 답변형식 지시
system_message_content = "귀하는 반도체 공정을 전문으로 하는 친절한 보조원입니다. 귀하의 주요 목표는 사용자에게 정확하고 자세한 정보를 제공하는 것입니다. 모든 최종 답변은 한국어로 명확하고 정중하게 작성해 주시기 바랍니다."
agent_kwargs = {"system_message": SystemMessage(content=system_message_content)}

In [9]:
%%time
from langchain_core.messages import HumanMessage, SystemMessage

# 테스트를 위한 간단한 사용자 메시지
user_message = HumanMessage(content="반도체 공정에서 웨이퍼란 무엇인가요?")

# llm 호출하여 응답 테스트
response = llm.invoke([agent_kwargs["system_message"], user_message])

# 응답 출력
print(response.content)
# Wall time: 9.27 s

반도체 공정에서 웨이퍼(Wafer)는 반도체 소자를 제조하기 위한 기본 재료로, 일반적으로 실리콘(Silicon)으로 만들어진 얇고 평평한 원판 형태입니다. 웨이퍼는 반도체 소자의 기초가 되며, 다양한 전자 소자와 회로를 형성하는 데 사용됩니다.

웨이퍼의 주요 특징은 다음과 같습니다:

1. **재료**: 대부분의 웨이퍼는 고순도의 실리콘으로 제작되지만, 갈륨 비소(GaAs)나 실리콘 카바이드(SiC)와 같은 다른 반도체 재료도 사용될 수 있습니다.

2. **두께와 직경**: 웨이퍼는 일반적으로 200mm(8인치) 또는 300mm(12인치) 직경으로 제작되며, 두께는 약 0.5mm에서 1mm 정도입니다.

3. **제조 과정**: 웨이퍼는 단결정 실리콘을 성장시키고, 이를 얇게 절단하여 제조됩니다. 이 과정에서 웨이퍼의 표면은 매우 매끄럽고 균일해야 하며, 이는 후속 공정에서 중요한 역할을 합니다.

4. **공정 단계**: 웨이퍼는 포토리소그래피, 에칭, 도핑 등의 다양한 공정을 거쳐 반도체 소자와 회로가 형성됩니다. 이러한 공정은 웨이퍼의 표면에 미세한 패턴을 생성하여 전자 소자의 기능을 구현합니다.

웨이퍼는 반도체 산업에서 매우 중요한 요소로, 전자기기와 컴퓨터의 핵심 부품인 반도체 소자의 생산에 필수적입니다.
CPU times: user 28.5 ms, sys: 3.88 ms, total: 32.3 ms
Wall time: 10.7 s


## LangChain Agent 생성 (ReAct Agent)  
> agent: 사용할 에이전트의 유형을 지정합니다. (e.g., ZERO_SHOT_REACT_DESCRIPTION)   
 tools: 에이전트가 사용할 도구 목록  
 llm: 에이전트의 추론을 담당할 언어 모델  
 verbose=True: 에이전트의 생각(Thought)과 행동(Action) 과정을 모두 출력하여 디버깅에 용이하게 합니다.

**AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION**

>ReAct 루프를 채팅 모델로 수행 :  
 모델이 Thought → Action → Action Input → Observation ... → Final Answer의 형식으로 추론/행동을 번갈아 수행  
>툴 설명(Descriptions)을 프롬프트에 주입하고, 모델은 이 설명을 바탕으로 어떤 툴을 언제 쓸지 스스로 결정함  
> Zero-shot이라 예시 샘플(few-shot)을 기본 포함하지 않으며, 메모리는 기본 내장하지 않음

>에이전트는 내부적으로 프롬프트 템플릿을 하나 만들고, 거기에:  
사용할 수 있는 Tool 목록  
각 Tool의 이름과 설명  
“어떻게 Tool을 선택하고 호출해야 하는지”에 대한 규칙  
을 문자열로 넣어서 LLM에게 전달합니다.

In [10]:
# LangChain Agent 생성
# 채팅 모델에 더 적합한 'CHAT_ZERO_SHOT_REACT_DESCRIPTION' 에이전트 사용 및 kwargs 전달
agent_final_korean = initialize_agent(
    tools,                                            # LLM이 사용가능한 Tool정보
    llm,                                              # Agent가 사용할 LLM
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION, # ReAct Prompt 적용
    verbose=True,                                     # ReAct 과정을 출력
    handle_parsing_errors=True,
    agent_kwargs=agent_kwargs,                        # 시스템 프롬프트 전달
    max_iterations=4,)                                # (선택): 루프 제한

/tmp/ipython-input-2462977590.py:3: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent_final_korean = initialize_agent(


## 검증 쿼리 실행  

In [11]:
# 검증 쿼리 실행
agent_final_korean.verbose = True  # 중간 과정 설정
print("--- 검증 :  단일 Tool 테스트 ---")
response = agent_final_korean.invoke("식각 공정의 상세 정보와 측정 단위를 알려줘.")
print("\n최종 답변 :")
print(response)

--- 검증 :  단일 Tool 테스트 ---


> Entering new AgentExecutor chain...
Thought: 식각 공정에 대한 상세 정보와 측정 단위를 찾기 위해 Semiconductor Process DB Lookup 도구를 사용해야겠다.
Action:
```
{
  "action": "Semiconductor Process DB Lookup",
  "action_input": "식각"
}
```
🔍 [고급 검색] '식각' 분석 중...
[검색 성공] 식각 공정 정보 발견

Observation: 
====== 식각 공정 상세 정보 ======
- 주요 파라미터: 가스 유량, 압력(Pressure), RF 파워, 식각 시간
- 측정 단위: 가스유량(sccm), 압력(mTorr), RF파워(W), 시간(min)
- 공정 설명: 불필요한 물질을 제거하여 원하는 패턴을 만드는 공정
- 중요도: 선폭 제어와 프로파일 형성에 핵심적

Thought:I now know the final answer.
Final Answer: 식각 공정의 상세 정보는 다음과 같습니다:
- 주요 파라미터: 가스 유량, 압력(Pressure), RF 파워, 식각 시간
- 측정 단위: 가스유량(sccm), 압력(mTorr), RF파워(W), 시간(min)
- 공정 설명: 불필요한 물질을 제거하여 원하는 패턴을 만드는 공정
- 중요도: 선폭 제어와 프로파일 형성에 핵심적

> Finished chain.

최종 답변 :
{'input': '식각 공정의 상세 정보와 측정 단위를 알려줘.', 'output': '식각 공정의 상세 정보는 다음과 같습니다:\n- 주요 파라미터: 가스 유량, 압력(Pressure), RF 파워, 식각 시간\n- 측정 단위: 가스유량(sccm), 압력(mTorr), RF파워(W), 시간(min)\n- 공정 설명: 불필요한 물질을 제거하여 원하는 패턴을 만드는 공정\n- 중요도: 선폭 제어와 프로파일 형성에 핵심적'}


최종 답변 :  
{'input': '식각 공정의 상세 정보와 측정 단위를 알려줘.',   
'output': '식각 공정의 상세 정보는 다음과 같습니다:\n
  - 주요 파라미터: 가스 유량, 압력(Pressure), RF 파워, 식각 시간\n
  - 측정 단위: 가스유량(sccm), 압력(mTorr), RF파워(W), 시간(min)\n
  - 공정 설명: 불필요한 물질을 제거하여 원하는 패턴을 만드는 공정\n
  - 중요도: 선폭 제어와 프로파일 형성에 핵심적'}



In [12]:
# 검증 쿼리 실행
agent_final_korean.verbose = False  # 중간 과정 설정
print("--- 검증 :  단일 Tool 테스트 ---")
response = agent_final_korean.invoke("식각 공정의 상세 정보와 측정 단위를 알려줘.")
print("\n최종 답변 :")
print(response)

--- 검증 :  단일 Tool 테스트 ---
🔍 [고급 검색] '식각' 분석 중...
[검색 성공] 식각 공정 정보 발견

최종 답변 :
{'input': '식각 공정의 상세 정보와 측정 단위를 알려줘.', 'output': '식각 공정의 상세 정보는 다음과 같습니다:\n- 주요 파라미터: 가스 유량, 압력(Pressure), RF 파워, 식각 시간\n- 측정 단위: 가스유량(sccm), 압력(mTorr), RF파워(W), 시간(min)\n- 공정 설명: 불필요한 물질을 제거하여 원하는 패턴을 만드는 공정\n- 중요도: 선폭 제어와 프로파일 형성에 핵심적'}


In [13]:
agent_final_korean.verbose = True   # 중간 과정 설정
print("--- 검증 :  단일 Tool 테스트 ---")
response = agent_final_korean.invoke("포토리소그래피 단계의 주요 파라미터를 알려줘.")
print("\n최종 답변 :")
print(response)

--- 검증 :  단일 Tool 테스트 ---


> Entering new AgentExecutor chain...
Thought: 포토리소그래피 단계의 주요 파라미터에 대한 정보를 찾기 위해 Semiconductor Process DB Lookup 도구를 사용해야겠다.
Action:
```
{
  "action": "Semiconductor Process DB Lookup",
  "action_input": "포토리소그래피"
}
```
🔍 [고급 검색] '포토리소그래피' 분석 중...
[검색 성공] 포토리소그래피 공정 정보 발견

Observation: 
====== 포토리소그래피 공정 상세 정보 ======
- 주요 파라미터: 노광량(Exposure Dose), 초점(Focus), 감광액(PR) 두께
- 측정 단위: 노광량(mJ/cm²), 초점(μm), PR두께(nm)
- 공정 설명: 반도체 웨이퍼 위에 회로 패턴을 형성하는 핵심 공정
- 중요도: 패턴 해상도와 정밀도를 결정하는 가장 중요한 공정

Thought:Thought: 포토리소그래피 단계의 주요 파라미터에 대한 정보를 이미 찾았으므로, 이를 정리하여 최종 답변을 작성해야겠다.
Final Answer: 포토리소그래피 단계의 주요 파라미터는 다음과 같습니다: 
- 노광량(Exposure Dose): mJ/cm²
- 초점(Focus): μm
- 감광액(PR) 두께: nm

이 공정은 반도체 웨이퍼 위에 회로 패턴을 형성하는 핵심 공정이며, 패턴 해상도와 정밀도를 결정하는 가장 중요한 공정입니다.

> Finished chain.

최종 답변 :
{'input': '포토리소그래피 단계의 주요 파라미터를 알려줘.', 'output': '포토리소그래피 단계의 주요 파라미터는 다음과 같습니다: \n- 노광량(Exposure Dose): mJ/cm²\n- 초점(Focus): μm\n- 감광액(PR) 두께: nm\n\n이 공정은 반도체 웨이퍼 위에 회로 패턴을 형성하는 핵심 공정이며, 패턴 해상도와 정밀도를 결

최종 답변 :  
{'input': '포토리소그래피 단계의 주요 파라미터를 알려줘.',   
'output': '포토리소그래피 단계의 주요 파라미터는 다음과 같습니다: 노광량(Exposure Dose), 초점(Focus), 감광액(PR) 두께입니다.   
측정 단위는 각각 노광량(mJ/cm²), 초점(μm), PR두께(nm)입니다.   
이 공정은 반도체 웨이퍼 위에 회로 패턴을 형성하는 핵심 공정이며, 패턴 해상도와 정밀도를 결정하는 가장 중요한 공정입니다.'}



In [14]:
# 검증 쿼리 실행 : 복합 질문 테스트
# 중간 과정 보기
agent_final_korean.verbose = True
print("\n\n--- 검증 : 복합 질문(Multi-Tool) 테스트 ---")
query_complex = "DUV 공정 매뉴얼에서 노광 단계 설명을 요약하고, 포토리소그래피 공정의 중요도도 알려줘."
response = agent_final_korean.invoke(query_complex)
print("\n최종 답변:")
print(response)



--- 검증 : 복합 질문(Multi-Tool) 테스트 ---


> Entering new AgentExecutor chain...
Thought: DUV 공정 매뉴얼에서 노광 단계에 대한 설명을 요약하고, 포토리소그래피 공정의 중요도를 확인하기 위해 매뉴얼을 로드해야 한다. 

Action:
```
{
  "action": "Manual Loader",
  "action_input": "DUV_manual.txt"
}
```

Observation: 
= DUV (Deep Ultraviolet) 공정 매뉴얼 =

문서 버전: 1.2
작성일: 2025-06-12

== 1. 개요 ==
DUV 리소그래피는 248nm 또는 193nm 파장의 빛을 사용하여 웨이퍼에 미세 회로 패턴을 형성하는 핵심 공정입니다.

== 2. 주요 단계 ==
2.1 웨이퍼 준비 (Wafer Preparation)
- 웨이퍼 세척 및 표면 처리

2.2 감광액 도포 (PR Coating)
- 스핀 코팅 방식을 사용하여 균일한 두께의 감광액(PR) 막을 형성합니다.

2.3 노광 (Exposure)
- 마스크에 설계된 패턴을 빛을 이용해 웨이퍼의 감광액 위로 전사시킵니다.
- **핵심 파라미터:** 노광 에너지(Dose), 초점(Focus)이 패턴의 정밀도를 결정합니다.
- 노광 장비의 정렬(Align) 정확도가 수율에 큰 영향을 미칩니다.

2.4 현상 (Development)
- 노광된 영역 또는 노광되지 않은 영역의 감광액을 선택적으로 제거하여 패턴을 완성합니다.

Thought:노광 단계에 대한 설명을 요약하였고, 포토리소그래피 공정의 중요도를 확인해야 한다. 포토리소그래피 공정의 중요도를 확인하기 위해 Semiconductor Process DB Lookup을 사용해야 한다.

Action:
```
{
  "action": "Semiconductor Process DB Lookup",
  "action_input": "포토리소그래피"
}
```
🔍 [고급 검색] '포토리소그래

최종 답변:  
{'input': 'DUV 공정 매뉴얼에서 노광 단계 설명을 요약하고, 포토리소그래피 공정의 중요도도 알려줘.',   
'output': '노광 단계에서는 마스크에 설계된 패턴을 빛을 이용해 웨이퍼의 감광액 위로 전사시키며, 노광 에너지와 초점이 패턴의 정밀도를 결정합니다. 포토리소그래피 공정은 반도체 웨이퍼 위에 회로 패턴을 형성하는 핵심 공정으로, 패턴 해상도와 정밀도를 결정하는 가장 중요한 공정입니다.'}



In [15]:
agent_final_korean.verbose = True
response = agent_final_korean.invoke("증착 공정의 중요도는?")
print("\n최종 답변:")
print(response)



> Entering new AgentExecutor chain...
Thought: 증착 공정에 대한 중요도를 확인하기 위해 Semiconductor Process DB Lookup 도구를 사용해야 합니다. 
Action:
```
{
  "action": "Semiconductor Process DB Lookup",
  "action_input": "증착"
}
```
🔍 [고급 검색] '증착' 분석 중...
[검색 성공] 증착 공정 정보 발견

Observation: 
====== 증착 공정 상세 정보 ======
- 주요 파라미터: 온도, 압력, 전구체(Precursor) 유량
- 측정 단위: 온도(°C), 압력(Torr), 유량(sccm)
- 공정 설명: 웨이퍼 위에 얇은 막을 형성하는 공정
- 중요도: 막질과 두께 균일성을 결정

Thought:증착 공정의 중요도에 대한 정보를 확인했습니다. 이 공정은 막질과 두께 균일성을 결정하는 데 중요한 역할을 합니다. 

Final Answer: 증착 공정의 중요도는 막질과 두께 균일성을 결정하는 것입니다.

> Finished chain.

최종 답변:
{'input': '증착 공정의 중요도는?', 'output': '증착 공정의 중요도는 막질과 두께 균일성을 결정하는 것입니다.'}


# HF Transformer Tool-Calling Agent 구현

In [ ]:
#!pip install langchain langchain-openai --quiet

In [ ]:
import os
import json
from google.colab import userdata
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [ ]:
# # Colab 비밀 기능에서 OpenAI API 키 가져오기
# os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [ ]:
# 매뉴얼 텍스트 파일
## manual_content 앞에서 정의된
## semiconductor_processes = {... } 앞에서 정의된

## Tool로 사용할 함수 정의 (@tool 데코레이터 사용)  
 - tools list로 정의


In [ ]:
from langchain_core.tools import tool # Tool 생성을 위한 데코레이터

# Tool로 사용할 함수 정의 (@tool 데코레이터 사용)
# (@tool) 데코레이터를 사용하면 함수 자체가 LangChain Tool 객체로 변환됨
# 함수의 docstring이 자동으로 tool의 description이 되어 매우 편리합니다.
@tool
def process_lookup(query: str) -> str:
    """사용자 질문에서 '포토리소그래피', '식각', '증착' 같은 공정 이름을 찾아, 해당 공정의 상세 정보(파라미터, 설명, 단위, 중요도)를 반환합니다."""
    for process_name, info in semiconductor_processes.items():
        if process_name in query:
            return json.dumps(info, ensure_ascii=False) # 결과를 JSON 문자열로 반환하여 모델이 파싱하기 쉽게 함
    return "해당 공정을 찾을 수 없습니다."

@tool
def load_manual(path: str) -> str:
    """'DUV_manual.txt'와 같이 로컬 파일 시스템에 있는 텍스트 형식의 매뉴얼을 읽어 그 내용을 문자열로 반환합니다."""
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        return f"오류: '{path}' 파일을 찾을 수 없습니다."

# 도구 리스트 정의
tools = [process_lookup, load_manual]

## LLM Wrapper , Agent 생성

In [ ]:
#llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
# 프롬프트 템플릿 설정
# openai tool-calling을 잘 수행하도록 역할과 지시사항을 상세하게 설정
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 반도체 공정 전문가 AI 어시스턴트다. 사용자의 질문을 해결하기 위해 주어진 도구를 적극적으로 사용해야 한다. 모든 최종 답변은 반드시 상세하고 친절한 한국어로 작성해야 한다."),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"), # 에이전트의 중간 작업 내역(생각, 도구 호출 등)을 위한 자리
])

In [ ]:
# LangChain Agent 생성
# 채팅 모델에 더 적합한 'CHAT_ZERO_SHOT_REACT_DESCRIPTION' 에이전트 사용 및 kwargs 전달
agent_tool_calling = initialize_agent(
    tools,
    llm,
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION, # 채팅 모델에 최적화된 ReAct 에이전트
    verbose=True,
    handle_parsing_errors=True,
    agent_kwargs=agent_kwargs ) # 시스템 프롬프트 전달

# 에이전트 실행 및 검증


In [ ]:
# 에이전트 실행 및 검증
print("\n\n--- 복합 질문(Multi-Tool) 테스트 ---")
complex_query = "DUV 공정 매뉴얼에서 노광 단계 설명을 요약하고, 포토리소그래피 공정의 중요도도 알려줘."

# .invoke()를 사용하여 에이전트 실행
agent_tool_calling.verbose = False  #
response = agent_tool_calling.invoke({"input": complex_query})

print("\n\n최종 답변:")
print(response['output'])



--- 복합 질문(Multi-Tool) 테스트 ---


최종 답변:
DUV 공정의 노광 단계는 마스크에 설계된 패턴을 빛을 이용해 웨이퍼의 감광액 위로 전사시키는 과정입니다. 이 단계에서 핵심 파라미터는 노광 에너지와 초점이며, 장비의 정렬 정확도가 수율에 큰 영향을 미칩니다. 포토리소그래피 공정은 반도체 웨이퍼 위에 회로 패턴을 형성하는 핵심 공정으로, 패턴 해상도와 정밀도를 결정하는 가장 중요한 공정입니다.
